Week 2
Day 6 Feature Engineering and Encoding

In [1]:
# Import libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer


#create the dataset
data = {
    "Age": [22, 25, 28, 35, 40],
    "Salary": [25000, 40000, 50000, 70000, 90000],
    "City": ["Delhi", "Mumbai", "Delhi", "Chennai", "Mumbai"],
    "Gender": ["Male", "Female", "Male", "Female", "Male"],
    "Purchased": ["No", "Yes", "No", "Yes", "Yes"]
}

df = pd.DataFrame(data)

print("Original Dataset:")
print(df)



print("\nDataset Information:")
print(df.info())

print("\nCategorical Columns:")
print(df.select_dtypes(include="object").columns)

#feature engineering

# Create new feature: Age Group
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 25, 35, 100],
    labels=["Young", "Adult", "Senior"]
)

# Create new feature: Salary in Thousands
df["Salary_k"] = df["Salary"] / 1000

print("\nAfter Feature Engineering:")
print(df)


#label encoding
encoder = LabelEncoder()

df["Gender_Encoded"] = encoder.fit_transform(df["Gender"])

print("\nAfter Label Encoding:")
print(df)


# one hot encoding
df_encoded = pd.get_dummies(
    df,
    columns=["City", "Age_Group"],
    drop_first=True
)

print("\nAfter One Hot Encoding:")
print(df_encoded)


#feature scaling
scaler = StandardScaler()

df_encoded[["Age", "Salary", "Salary_k"]] = scaler.fit_transform(
    df_encoded[["Age", "Salary", "Salary_k"]]
)

print("\nAfter Feature Scaling:")
print(df_encoded)

Original Dataset:
   Age  Salary     City  Gender Purchased
0   22   25000    Delhi    Male        No
1   25   40000   Mumbai  Female       Yes
2   28   50000    Delhi    Male        No
3   35   70000  Chennai  Female       Yes
4   40   90000   Mumbai    Male       Yes

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Age        5 non-null      int64 
 1   Salary     5 non-null      int64 
 2   City       5 non-null      object
 3   Gender     5 non-null      object
 4   Purchased  5 non-null      object
dtypes: int64(2), object(3)
memory usage: 332.0+ bytes
None

Categorical Columns:
Index(['City', 'Gender', 'Purchased'], dtype='object')

After Feature Engineering:
   Age  Salary     City  Gender Purchased Age_Group  Salary_k
0   22   25000    Delhi    Male        No     Young      25.0
1   25   40000   Mumbai  Female       Yes     

Day 7 Feature Scaling and selection

In [2]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.linear_model import LogisticRegression


#dataset
data = {
    "Age": [22, 25, 28, 35, 40, 45],
    "Salary": [25000, 40000, 50000, 70000, 90000, 100000],
    "Experience": [1, 3, 5, 8, 12, 15],
    "Score": [70, 75, 80, 85, 90, 95],
    "Purchased": [0, 0, 1, 1, 1, 1]
}

df = pd.DataFrame(data)

print("Original Dataset:")
print(df)

# Step 2: Separate Features and Target
X = df.drop("Purchased", axis=1)
y = df["Purchased"]

print("\nFeatures:")
print(X)

print("\nTarget:")
print(y)


# Step 3: Standard Scaling
scaler = StandardScaler()

X_standard = scaler.fit_transform(X)

X_standard = pd.DataFrame(
    X_standard,
    columns=X.columns
)

print("\nAfter Standard Scaling:")
print(X_standard)

# Step 4: Min-Max Scaling
minmax = MinMaxScaler()

X_minmax = minmax.fit_transform(X)

X_minmax = pd.DataFrame(
    X_minmax,
    columns=X.columns
)

print("\nAfter Min-Max Scaling:")
print(X_minmax)


# Step 5: Feature Selection using SelectKBest

selector = SelectKBest(
    score_func=chi2,
    k=2
)

# Chi-square requires non-negative values
X_positive = MinMaxScaler().fit_transform(X)

X_selected = selector.fit_transform(
    X_positive,
    y
)

selected_features = X.columns[
    selector.get_support()
]

print("\nSelected Features:")
print(selected_features)

print("\nFeature Scores:")
print(
    pd.DataFrame({
        "Feature": X.columns,
        "Score": selector.scores_
    })
)


# feature slection using RFE

model = LogisticRegression()

rfe = RFE(
    estimator=model,
    n_features_to_select=2
)

rfe.fit(X, y)

print("\nRFE Selected Features:")
print(
    X.columns[rfe.support_]
)

print("\nRFE Ranking:")
print(
    pd.DataFrame({
        "Feature": X.columns,
        "Ranking": rfe.ranking_
    })
)

Original Dataset:
   Age  Salary  Experience  Score  Purchased
0   22   25000           1     70          0
1   25   40000           3     75          0
2   28   50000           5     80          1
3   35   70000           8     85          1
4   40   90000          12     90          1
5   45  100000          15     95          1

Features:
   Age  Salary  Experience  Score
0   22   25000           1     70
1   25   40000           3     75
2   28   50000           5     80
3   35   70000           8     85
4   40   90000          12     90
5   45  100000          15     95

Target:
0    0
1    0
2    1
3    1
4    1
5    1
Name: Purchased, dtype: int64

After Standard Scaling:
        Age    Salary  Experience    Score
0 -1.277231 -1.402829   -1.286842 -1.46385
1 -0.912308 -0.841698   -0.880471 -0.87831
2 -0.547385 -0.467610   -0.474100 -0.29277
3  0.304103  0.280566    0.135457  0.29277
4  0.912308  1.028741    0.948200  0.87831
5  1.520513  1.402829    1.557757  1.46385

After Min-

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Day 8 Handling imbalanced data

In [3]:
# Import libraries
import pandas as pd
import numpy as np

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from imblearn.over_sampling import SMOTE
from collections import Counter


# -----------------------------------
# Step 1: Create Imbalanced Dataset
# -----------------------------------

X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_classes=2,
    weights=[0.90, 0.10],
    random_state=42
)

print("Original Class Distribution:")
print(Counter(y))


# Convert into DataFrame
df = pd.DataFrame(X, columns=[
    "Feature1",
    "Feature2",
    "Feature3",
    "Feature4",
    "Feature5"
])

df["Target"] = y

print("\nDataset:")
print(df.head())




Original Class Distribution:
Counter({np.int64(0): 895, np.int64(1): 105})

Dataset:
   Feature1  Feature2  Feature3  Feature4  Feature5  Target
0 -0.439643  0.542547 -0.822420  0.401366 -0.854840       0
1 -1.320268 -0.451656 -1.147691  0.217991  2.515569       0
2 -0.902414 -0.301790 -2.084113  0.152282  1.702509       0
3 -1.658183  1.183085  1.112688  1.104253 -1.115765       0
4 -1.598711  0.169268 -0.926698  0.603763  1.296845       0


In [4]:
# -----------------------------------
# Step 2: Split Features and Target
# -----------------------------------

X = df.drop("Target", axis=1)
y = df["Target"]




In [5]:
# -----------------------------------
# Step 3: Train-Test Split
# -----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("\nBefore SMOTE:")
print(Counter(y_train))


# -----------------------------------
# Step 4: Apply SMOTE
# -----------------------------------

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)


print("\nAfter SMOTE:")
print(Counter(y_train_smote))


# -----------------------------------
# Step 5: Feature Scaling
# -----------------------------------

scaler = StandardScaler()

X_train_smote = scaler.fit_transform(X_train_smote)
X_test = scaler.transform(X_test)


# -----------------------------------
# Step 6: Train Machine Learning Model
# -----------------------------------

model = LogisticRegression()

model.fit(
    X_train_smote,
    y_train_smote
)


# -----------------------------------
# Step 7: Model Evaluation
# -----------------------------------

y_pred = model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Before SMOTE:
Counter({0: 710, 1: 90})

After SMOTE:
Counter({0: 710, 1: 710})

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.89      0.94       185
           1       0.39      0.87      0.54        15

    accuracy                           0.89       200
   macro avg       0.69      0.88      0.74       200
weighted avg       0.94      0.89      0.91       200



Day 9 Test train and split

In [6]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [7]:
# Load Iris dataset
iris = load_iris()

# Create DataFrame
df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df["target"] = iris.target

print("Dataset:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

Dataset:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  

Dataset Shape:
(150, 5)


In [8]:
# Features
X = df.drop("target", axis=1)

# Target
y = df["target"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2
3                4.6               3.1                1.5               0.2
4                5.0               3.6                1.4               0.2

Target:
0    0
1    0
2    0
3    0
4    0
Name: target, dtype: int64


In [9]:
# Split dataset into training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:")
print(X_train.shape)

print("\nTesting Data:")
print(X_test.shape)

Training Data:
(120, 4)

Testing Data:
(30, 4)


In [10]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [11]:
# Create model
model = LogisticRegression()

# Train model
model.fit(
    X_train,
    y_train
)

LogisticRegression()

In [12]:
# Prediction
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


In [13]:
# Create K-Fold
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Perform cross-validation
scores = cross_val_score(
    model,
    X,
    y,
    cv=kfold
)

print("Cross Validation Scores:")
print(scores)

print("\nAverage CV Accuracy:")
print(scores.mean())

Cross Validation Scores:
[1.         1.         0.93333333 0.96666667 0.96666667]

Average CV Accuracy:
0.9733333333333334
